In [1]:
# Load Data
import os
from langchain_community.document_loaders import PyPDFLoader, TextLoader, Docx2txtLoader

DATA_PATH = "data"

def load_documents():
    documents = []

    for file in os.listdir(DATA_PATH):
        path = os.path.join(DATA_PATH, file)

        if file.endswith(".pdf"):
            loader = PyPDFLoader(path)
        elif file.endswith(".txt"):
            loader = TextLoader(path)
        elif file.endswith(".docx"):
            loader = Docx2txtLoader(path)
        else:
            continue

        docs = loader.load()

        for doc in docs:
            doc.metadata["source"] = file

        documents.extend(docs)

    return documents

documents = load_documents()
print("Loaded:", len(documents))

Loaded: 6


In [2]:
# Chunking
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap

    return chunks

chunks = []
metadata = []

for doc in documents:
    split_chunks = chunk_text(doc.page_content)
    chunks.extend(split_chunks)
    metadata.extend([doc.metadata]*len(split_chunks))

print("Chunks:", len(chunks))

Chunks: 39


In [3]:
# Embeddings
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embed_model.encode(chunks, show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
# Store in chroma DB
import chromadb

client = chromadb.Client()
collection = client.create_collection(name="rag_collection")

for i, chunk in enumerate(chunks):
    collection.add(
        documents=[chunk],
        embeddings=[embeddings[i].tolist()],
        metadatas=[metadata[i]],
        ids=[str(i)]
    )

print("Vector DB ready ")

Vector DB ready 


In [5]:
# Retriever
def retrieve(query, k=3):
    query_embedding = embed_model.encode([query])[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=k
    )

    return results["documents"][0], results["metadatas"][0]

In [7]:
# lonading LLM Flan
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_length=256
)

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

C:\Users\Anoop\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Anoop\.cache\huggingface\hub\models--google--flan-t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'Diff

In [8]:
# Final RAG 
def ask_question(query):
    docs, metas = retrieve(query)

    if not docs:
        return "I don't know.", []

    context = "\n\n".join(docs)

    prompt = f"""
You are a strict question answering system.

Rules:
- Answer ONLY from the context
- Do NOT guess
- If answer is not present, say "I don't know"

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm(prompt)[0]["generated_text"]

    return response, metas

In [9]:
# Test run
query = "What caused the French Revolution?"

answer, sources = ask_question(query)

print("Answer:\n", answer)

print("\nSources:")
for s in sources:
    print(s)

Answer:
 
You are a strict question answering system.

Rules:
- Answer ONLY from the context
- Do NOT guess
- If answer is not present, say "I don't know"

Context:
The French Revolution, which began in 1789, was one of the most significant events in modern history, fundamentally transforming France’s political, social, and economic structure. It marked the end of absolute monarchy, challenged centuries-old feudal privileges, and introduced ideals such as liberty, equality, and fraternity. These principles not only reshaped France but also influenced democratic movements across the world.

Before the revolution, France was governed by an absolute monarchy u

tion, bore the burden of heavy taxes. This imbalance created widespread resentment and dissatisfaction.

Economic crisis played a crucial role in triggering the revolution. France was heavily in debt due to its involvement in wars, including the American Revolution. Additionally, poor harvests led to food shortages, causing bread p

In [10]:
# Interaction
while True:
    query = input("\nAsk (type exit to quit): ")

    if query.lower() == "exit":
        break

    answer, sources = ask_question(query)

    print("\nAnswer:\n", answer)


Ask (type exit to quit):  What were the main causes of the French Revolution?



Answer:
 
You are a strict question answering system.

Rules:
- Answer ONLY from the context
- Do NOT guess
- If answer is not present, say "I don't know"

Context:
The French Revolution, which began in 1789, was one of the most significant events in modern history, fundamentally transforming France’s political, social, and economic structure. It marked the end of absolute monarchy, challenged centuries-old feudal privileges, and introduced ideals such as liberty, equality, and fraternity. These principles not only reshaped France but also influenced democratic movements across the world.

Before the revolution, France was governed by an absolute monarchy u

tion, bore the burden of heavy taxes. This imbalance created widespread resentment and dissatisfaction.

Economic crisis played a crucial role in triggering the revolution. France was heavily in debt due to its involvement in wars, including the American Revolution. Additionally, poor harvests led to food shortages, causing bread 


Ask (type exit to quit):  9.	What strategies did leaders use in the Indian Independence Movement? 



Answer:
 
You are a strict question answering system.

Rules:
- Answer ONLY from the context
- Do NOT guess
- If answer is not present, say "I don't know"

Context:
The Indian Independence Movement was a prolonged struggle against British colonial rule, culminating in Indiaâ€™s independence in 1947. It was characterized by a combination of political activism, mass movements, and ideological diversity.

British rule in India began with the East India Company and later transitioned to direct control by the British Crown. Over time, economic exploitation, cultural interference, and political repression led to growing discontent among Indians.

The Indian Natio

ooperation Movement, Civil Disobedience Movement, and Quit India Movement mobilized millions of people.

Other leaders, such as Subhas Chandra Bose, advocated for armed struggle. The movement saw participation from various sections of society, including women and youth.

India achieved independence on August 15, 1947. However, the


Ask (type exit to quit):  exit
